In [1]:
import xgboost
import json

/home/users/iareed/miniconda3/envs/higgs-dna/lib/python3.10/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index


In [4]:
with open('data/2HDM_M250_BDT.json') as f_in:
    training_features = json.load(f_in)['training_features']

In [63]:
training_features[57]

'lepton_3_mass'

In [7]:
fixed = xgboost.Booster()
broken = xgboost.Booster()

In [44]:
fixed.load_model('output/2HDM_M250_11Aug22.xgb')
broken.load_model('output/2HDM_M250_11Aug22_broken_cand.xgb')

In [56]:
broken.get_score(importance_type='gain')

{'f0': 10.928776741027832,
 'f1': 15.407617568969727,
 'f2': 25.457368850708008,
 'f3': 17.996623992919922,
 'f4': 22.06524658203125,
 'f5': 15.873287200927734,
 'f6': 14.806896209716797,
 'f7': 10.675594329833984,
 'f8': 64.54082489013672,
 'f9': 46.05803298950195,
 'f10': 28.207162857055664,
 'f11': 35.81821060180664,
 'f12': 11.229248046875,
 'f13': 1.4408493041992188,
 'f14': 3.7122139930725098,
 'f16': 25.126169204711914,
 'f17': 42.527252197265625,
 'f18': 48.31004333496094,
 'f20': 52.60420227050781,
 'f21': 84.38494873046875,
 'f22': 14.90333366394043,
 'f23': 10.950915336608887,
 'f24': 9.313652038574219,
 'f25': 12.571310043334961,
 'f26': 15.960783958435059,
 'f27': 9.000880241394043,
 'f28': 10.884471893310547,
 'f29': 61.48105239868164,
 'f30': 11.438152313232422,
 'f31': 7.023202896118164,
 'f32': 60.80386734008789,
 'f33': 17.487428665161133,
 'f34': 7.143673896789551,
 'f35': 147.21884155273438,
 'f36': 7.657928466796875,
 'f37': 10.68062973022461,
 'f38': 59.7508239746

In [57]:
print(broken.get_score(importance_type='gain').keys())
print(len(broken.get_score(importance_type='gain').keys()))


dict_keys(['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f14', 'f16', 'f17', 'f18', 'f20', 'f21', 'f22', 'f23', 'f24', 'f25', 'f26', 'f27', 'f28', 'f29', 'f30', 'f31', 'f32', 'f33', 'f34', 'f35', 'f36', 'f37', 'f38', 'f39', 'f40', 'f41', 'f42', 'f43', 'f44', 'f45', 'f46', 'f47', 'f50', 'f51', 'f52', 'f55', 'f60', 'f61', 'f62', 'f63', 'f65', 'f66', 'f67', 'f68', 'f69', 'f70', 'f71'])
61


In [74]:
def calc_importance(features, model):
    dummy_names = []
    i=0
    while i < len(features):
        dummy_names.append('f{}'.format(i))
        i+=1
    tmp_weights = model.get_score(importance_type='gain')
    tmp = []
    keys = tmp_weights.keys()
    for key in dummy_names:
        if key not in keys:
            tmp.append(0.0)
            continue
        tmp.append(tmp_weights[key])
    tmp_dir = dict(zip(features,tmp))
    return tmp_dir

In [77]:
fixed_weights = calc_importance(training_features,fixed)
broken_weights = calc_importance(training_features,broken)

In [82]:
for key in training_features:
    print('feature: {},     {},                 {}'.format(key, fixed_weights[key], broken_weights[key]))

feature: Diphoton_eta,     9.069828033447266,                 10.928776741027832
feature: Diphoton_pt_mgg,     15.306122779846191,                 15.407617568969727
feature: Diphoton_dR,     25.993268966674805,                 25.457368850708008
feature: Diphoton_helicity,     20.85222625732422,                 17.996623992919922
feature: LeadPhoton_mvaID,     24.04030990600586,                 22.06524658203125
feature: SubleadPhoton_mvaID,     17.870811462402344,                 15.873287200927734
feature: LeadPhoton_eta,     11.651350975036621,                 14.806896209716797
feature: SubleadPhoton_eta,     13.382277488708496,                 10.675594329833984
feature: LeadPhoton_pixelSeed,     69.33394622802734,                 64.54082489013672
feature: SubleadPhoton_pixelSeed,     54.6926383972168,                 46.05803298950195
feature: LeadPhoton_pt_mgg,     25.076860427856445,                 28.207162857055664
feature: SubleadPhoton_pt_mgg,     38.80842590332031,     

In [78]:
broken_weights

{'Diphoton_eta': 10.928776741027832,
 'Diphoton_pt_mgg': 15.407617568969727,
 'Diphoton_dR': 25.457368850708008,
 'Diphoton_helicity': 17.996623992919922,
 'LeadPhoton_mvaID': 22.06524658203125,
 'SubleadPhoton_mvaID': 15.873287200927734,
 'LeadPhoton_eta': 14.806896209716797,
 'SubleadPhoton_eta': 10.675594329833984,
 'LeadPhoton_pixelSeed': 64.54082489013672,
 'SubleadPhoton_pixelSeed': 46.05803298950195,
 'LeadPhoton_pt_mgg': 28.207162857055664,
 'SubleadPhoton_pt_mgg': 35.81821060180664,
 'ditau_pt': 11.229248046875,
 'ditau_eta': 1.4408493041992188,
 'ditau_mass': 3.7122139930725098,
 'ditau_dR': 0.0,
 'ditau_helicity': 25.126169204711914,
 'MET_pt': 42.527252197265625,
 'n_jets': 48.31004333496094,
 'n_leptons': 0.0,
 'n_electrons': 52.60420227050781,
 'n_muons': 84.38494873046875,
 'n_taus': 14.90333366394043,
 'jet_1_pt': 10.950915336608887,
 'jet_1_eta': 9.313652038574219,
 'jet_1_btagDeepFlavB': 12.571310043334961,
 'jet_2_pt': 15.960783958435059,
 'jet_2_eta': 9.000880241394